# Task 1: BestBuy Canada Review Scraping + Sentiment Analysis
Steps: setup -> scrape (Playwright + BeautifulSoup, pagination + sort filters) -> clean -> sentiment -> aspect tags -> export CSV.

**Important:** BestBuy changes its HTML often. Open the product page in Chrome DevTools (F12) and verify the CSS selectors in the `SELECTORS` dict before running.

In [ ]:
# !pip install playwright beautifulsoup4 pandas vaderSentiment transformers torch tqdm
# !playwright install chromium

In [ ]:
import re, time, uuid, random, json
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout
from tqdm.auto import tqdm

# ====== CONFIG ======
PRODUCT_URL = "https://www.bestbuy.ca/en-ca/product/REPLACE-WITH-YOUR-PRODUCT/SKU"   # product with >= 50 reviews
SORT_OPTIONS = ["Most Helpful", "Newest", "Highest Rating", "Lowest Rating"]            # labels as shown on the site
MAX_SHOW_MORE_CLICKS = 40
HEADLESS = False          # False is less likely to be flagged as a bot
PROXY = None              # e.g. {"server": "http://host:port", "username": "u", "password": "p"}
SOURCE_PLATFORM = "BestBuy Canada"

# Verify these in DevTools - adjust if site markup changed
SELECTORS = {
    "review_block": "[data-automation='review'], [class*='reviewItem'], [class*='review_']",
    "title": "[class*='reviewTitle'], [class*='review-title'], h3, h4",
    "body": "[class*='reviewContent'], [class*='review-content'], [class*='reviewText'], p",
    "rating": "[class*='rating'], [aria-label*='out of 5'], [aria-label*='stars']",
    "date": "time, [class*='date'], [class*='Date']",
    "author": "[class*='author'], [class*='reviewer'], [class*='nickname'], [class*='userName']",
    "show_more": "button:has-text('Show More'), button:has-text('Load More'), a:has-text('Show More')",
}

## Anti-scraping helpers: throttling, rotating User-Agents, retry with exponential backoff

In [ ]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
]

def polite_sleep(a=1.5, b=3.5):
    time.sleep(random.uniform(a, b))          # request throttling with jitter

def retry(fn, tries=4, base=2.0):
    for i in range(tries):
        try:
            return fn()
        except Exception as e:
            wait = base ** i + random.random()  # exponential backoff + jitter
            print(f"  retry {i+1}/{tries} after error: {type(e).__name__}; sleeping {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError("All retries failed")

## Scraper

In [ ]:
def parse_rating(node):
    if node is None: return None
    txt = " ".join([node.get_text(" ", strip=True), node.get("aria-label", "") or ""])
    m = re.search(r"(\d(?:\.\d)?)\s*(?:out of|/)\s*5", txt)
    if m: return float(m.group(1))
    # fallback: count filled star elements
    filled = len(node.select("[class*='filled'], [class*='full']"))
    return float(filled) if filled else None

def parse_date(text):
    if not text: return None
    text = text.strip()
    for fmt in ("%B %d, %Y", "%b %d, %Y", "%Y-%m-%d", "%d %B %Y", "%m/%d/%Y"):
        try: return datetime.strptime(text, fmt).strftime("%Y-%m-%d")
        except ValueError: pass
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})", text)
    return m.group(0) if m else None     # relative dates ("2 days ago") need manual handling

def parse_reviews(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for blk in soup.select(SELECTORS["review_block"]):
        g = lambda k: blk.select_one(SELECTORS[k])
        title, body, auth, dt = g("title"), g("body"), g("author"), g("date")
        date_raw = (dt.get("datetime") if dt is not None and dt.has_attr("datetime") else (dt.get_text(strip=True) if dt else None))
        rows.append({
            "review_id": str(uuid.uuid4()),
            "review_title": title.get_text(" ", strip=True) if title else "",
            "review_text": body.get_text(" ", strip=True) if body else "",
            "date": parse_date(date_raw),
            "star_rating": parse_rating(g("rating")),
            "source_platform": SOURCE_PLATFORM,
            "reviewer_name": auth.get_text(" ", strip=True) if auth else "Anonymous",
        })
    return rows

def select_sort(page, label):
    try:
        page.get_by_text(re.compile(r"sort", re.I)).first.click(timeout=3000)
        page.get_by_text(label, exact=False).first.click(timeout=3000)
        page.wait_for_load_state("networkidle"); polite_sleep()
        return True
    except Exception:
        # sort control may be a <select>
        try:
            page.select_option("select", label=label, timeout=3000); polite_sleep(); return True
        except Exception:
            print(f"  could not apply sort '{label}' - check selector"); return False

def expand_all(page):
    for i in range(MAX_SHOW_MORE_CLICKS):
        btn = page.locator(SELECTORS["show_more"]).first
        try:
            if not btn.is_visible(timeout=2000): break
            btn.scroll_into_view_if_needed(); btn.click(); polite_sleep(1, 2.5)
        except PWTimeout:
            break

def scrape(url):
    all_rows = []
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=HEADLESS, proxy=PROXY,
                                    args=["--disable-blink-features=AutomationControlled"])
        ctx = browser.new_context(user_agent=random.choice(USER_AGENTS), locale="en-CA",
                                  viewport={"width": 1366, "height": 900})
        ctx.add_init_script("Object.defineProperty(navigator,'webdriver',{get:()=>undefined})")  # basic stealth
        page = ctx.new_page()
        retry(lambda: page.goto(url, wait_until="domcontentloaded", timeout=60000))
        polite_sleep(3, 5)
        # If a CAPTCHA appears, solve it manually in the visible browser, then continue:
        # input("Solve CAPTCHA if shown, then press Enter...")
        try: page.get_by_text(re.compile(r"reviews", re.I)).first.scroll_into_view_if_needed(timeout=5000)
        except Exception: pass
        for label in tqdm(SORT_OPTIONS, desc="sort filters"):
            select_sort(page, label)
            expand_all(page)
            rows = parse_reviews(page.content())
            print(f"  [{label}] parsed {len(rows)} review blocks")
            all_rows += rows
        browser.close()
    return pd.DataFrame(all_rows)

raw_df = scrape(PRODUCT_URL)
raw_df.shape

## Deduplicate (same review appears under several sort orders) and clean

In [ ]:
def clean_text(s):
    s = re.sub(r"<[^>]+>", " ", str(s))
    s = re.sub(r"[^\x00-\x7F]+", " ", s)       # drop emojis / odd characters
    return re.sub(r"\s+", " ", s).strip()

df = raw_df.copy()
df["review_title"] = df["review_title"].map(clean_text)
df["review_text"] = df["review_text"].map(clean_text)
df = df[(df.review_text != "") | (df.review_title != "")]
df = df.drop_duplicates(subset=["reviewer_name", "review_title", "review_text", "date"]).reset_index(drop=True)
df["review_id"] = ["BBC-" + str(i+1).zfill(5) for i in range(len(df))]   # stable primary key
df["full_text"] = (df.review_title + ". " + df.review_text).str.strip(". ")
print(len(df), "unique reviews"); df.head()

## Sentiment analysis (VADER baseline + optional Transformer)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

def vader_label(text):
    c = vader.polarity_scores(text)["compound"]
    return ("Positive" if c >= 0.05 else "Negative" if c <= -0.05 else "Neutral"), c

df[["sentiment_vader", "vader_score"]] = df.full_text.apply(lambda t: pd.Series(vader_label(t)))

USE_TRANSFORMER = True   # set False if no GPU / slow machine
if USE_TRANSFORMER:
    from transformers import pipeline
    clf = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                   truncation=True, max_length=256)
    preds = clf(df.full_text.tolist(), batch_size=16)
    df["sentiment"] = [p["label"].capitalize() for p in preds]
    df["sentiment_conf"] = [round(p["score"], 3) for p in preds]
else:
    df["sentiment"], df["sentiment_conf"] = df.sentiment_vader, df.vader_score.abs()

# sanity check vs star rating
print(pd.crosstab(df.star_rating, df.sentiment))

## Aspect tagging combined with sentiment

In [ ]:
ASPECTS = {
    "Good Design & Quality": ["design","build","quality","sturdy","sleek","premium","looks","durable","solid"],
    "Performance": ["fast","speed","performance","smooth","lag","slow","powerful","responsive"],
    "Battery & Power": ["battery","charge","charging","power","last","lasts"],
    "Sound / Display": ["sound","audio","bass","screen","display","picture","brightness","camera","mic"],
    "Ease of Use & Setup": ["easy","setup","set up","install","simple","intuitive","pair","app","connect"],
    "Value for Money": ["price","value","worth","cheap","expensive","deal","money"],
    "Durability / Defects": ["broke","broken","defect","stopped working","faulty","crack","fail","died","return"],
    "Customer Service & Delivery": ["service","support","shipping","delivery","warranty","refund","packaging"],
    "Satisfactory & Recommended": ["recommend","love","great","happy","satisfied","perfect","excellent","good"],
}

def tag_review(text, sentiment):
    t = text.lower()
    tags = [f"{asp} ({sentiment[:3]})" for asp, kws in ASPECTS.items() if any(k in t for k in kws)]
    return tags or [f"General ({sentiment[:3]})"]

df["aspect_tags"] = [tag_review(t, s) for t, s in zip(df.full_text, df.sentiment)]
df[["review_id","star_rating","sentiment","aspect_tags"]].head(10)

## Satisfaction drivers vs pain points (for the report)

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
print(df.sentiment.value_counts(normalize=True).round(3))
exploded = df.explode("aspect_tags")
drivers = Counter(exploded[exploded.sentiment == "Positive"].aspect_tags).most_common(8)
pains   = Counter(exploded[exploded.sentiment == "Negative"].aspect_tags).most_common(8)
print("Drivers:", drivers); print("Pain points:", pains)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
df.sentiment.value_counts().plot.bar(ax=ax[0], title="Sentiment distribution")
df.star_rating.value_counts().sort_index().plot.bar(ax=ax[1], title="Star ratings")
pd.Series(dict(drivers)).sort_values().plot.barh(ax=ax[2], title="Top satisfaction drivers")
plt.tight_layout(); plt.show()

## Export

In [ ]:
out = df[["review_id","review_title","review_text","date","star_rating","source_platform",
          "reviewer_name","sentiment","sentiment_conf","aspect_tags"]].copy()
out["aspect_tags"] = out.aspect_tags.map(lambda x: str(x))
out.to_csv("task1_bestbuy_reviews_sentiment.csv", index=False)
out.to_excel("task1_bestbuy_reviews_sentiment.xlsx", index=False)
print("saved", len(out), "rows")